In [ ]:
!pip install -q flwr[simulation]

In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

import flwr as fl

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cpu


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import kagglehub

path = kagglehub.dataset_download(
    "hasnainjaved/melanoma-skin-cancer-dataset-of-10000-images"
)

print(path)

Using Colab cache for faster access to the 'melanoma-skin-cancer-dataset-of-10000-images' dataset.
/kaggle/input/melanoma-skin-cancer-dataset-of-10000-images


In [6]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

In [7]:
base_path = os.path.join(path, "melanoma_cancer_dataset")

train_path = os.path.join(base_path, "train")
test_path  = os.path.join(base_path, "test")

train_dataset = datasets.ImageFolder(
    train_path,
    transform=transform
)

test_dataset = datasets.ImageFolder(
    test_path,
    transform=transform
)

In [8]:
num_clients = 3

client_data_size = len(train_dataset) // num_clients

lengths = [client_data_size] * num_clients

lengths[-1] += len(train_dataset) - sum(lengths)

client_datasets = random_split(
    train_dataset,
    lengths
)

print(lengths)

[3201, 3201, 3203]


In [9]:
client_loaders = []

for dataset in client_datasets:

    loader = DataLoader(
        dataset,
        batch_size=32,
        shuffle=True
    )

    client_loaders.append(loader)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [10]:
class SkinCancerCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128,256,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1,1))
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(256,128),
            nn.ReLU(),

            nn.Dropout(0.5),

            nn.Linear(128,2)
        )

    def forward(self,x):

        x = self.features(x)

        x = self.classifier(x)

        return x

In [11]:
def train(model, loader, epochs=1):

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )

    model.train()

    for epoch in range(epochs):

        running_loss = 0.0

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        print(
            f"Loss: {running_loss/len(loader):.4f}"
        )

In [12]:
def test(model, loader):

    criterion = nn.CrossEntropyLoss()

    model.eval()

    loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            batch_loss = criterion(outputs, labels)

            loss += batch_loss.item()

            _, predicted = torch.max(outputs,1)

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

    accuracy = 100 * correct / total

    return loss, accuracy

In [13]:
class FlowerClient(fl.client.NumPyClient):

    def __init__(self, model, trainloader):

        self.model = model
        self.trainloader = trainloader

    def get_parameters(self, config):

        return [
            val.cpu().numpy()
            for _, val in self.model.state_dict().items()
        ]

    def set_parameters(self, parameters):

        params_dict = zip(
            self.model.state_dict().keys(),
            parameters
        )

        state_dict = {
            k: torch.tensor(v)
            for k, v in params_dict
        }

        self.model.load_state_dict(
            state_dict,
            strict=True
        )

    def fit(self, parameters, config):

        self.set_parameters(parameters)

        train(
            self.model,
            self.trainloader,
            epochs=1
        )

        return (
            self.get_parameters(config),
            len(self.trainloader.dataset),
            {}
        )

    def evaluate(self, parameters, config):

        self.set_parameters(parameters)

        loss, accuracy = test(
            self.model,
            test_loader
        )

        return (
            float(loss),
            len(test_loader.dataset),
            {"accuracy": float(accuracy)}
        )

In [14]:
from flwr.common import Context

def client_fn(context: Context):

    cid = context.node_config["partition-id"]

    model = SkinCancerCNN().to(device)

    trainloader = client_loaders[int(cid)]

    return FlowerClient(
        model,
        trainloader
    ).to_client()

In [17]:
global_model = SkinCancerCNN().to(device)

In [18]:
def get_parameters(model):

    return [
        val.cpu().numpy()
        for _, val in model.state_dict().items()
    ]


def set_parameters(model, parameters):

    params_dict = zip(
        model.state_dict().keys(),
        parameters
    )

    state_dict = {
        k: torch.tensor(v)
        for k, v in params_dict
    }

    model.load_state_dict(
        state_dict,
        strict=True
    )

In [19]:
def evaluate_fn(server_round, parameters, config):

    global global_model

    set_parameters(
        global_model,
        parameters
    )

    loss, accuracy = test(
        global_model,
        test_loader
    )

    print(

        f"\nGLOBAL MODEL -> "

        f"Round {server_round} "

        f"Accuracy: {accuracy:.2f}%"

    )

    # Save metrics
    global_accuracies.append(accuracy)

    global_losses.append(loss)

    return loss, {
        "accuracy": accuracy
    }

In [ ]:
global_accuracies = []

global_losses = []


strategy = fl.server.strategy.FedAvg(

    fraction_fit=1.0,

    min_fit_clients=3,

    min_available_clients=3,

    evaluate_fn=evaluate_fn
)

In [21]:
history = fl.simulation.start_simulation(

    client_fn=client_fn,

    num_clients=3,

    config=fl.server.ServerConfig(
        num_rounds=5
    ),

    strategy=strategy,
)

	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
INFO :      Starting Flower simulation, config: num_rounds=5, no round_timeout
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
2026-05-03 07:01:49,211	INFO worker.py:2012 -- Started a local Ray instance.
INFO :      Flower VCE: Ray initialized with resources: {'CPU': 2.0, 'object_store_memory': 3983856844.0, 'node:172.28.0.12': 1.


GLOBAL MODEL -> Round 0 Accuracy: 50.00%
(ClientAppActor pid=23996) Loss: 0.5146
(ClientAppActor pid=23997) Loss: 0.4776


INFO :      aggregate_fit: received 3 results and 0 failures


(ClientAppActor pid=23996) Loss: 0.4922


INFO :      fit progress: (1, 14.85165587067604, {'accuracy': 80.7}, 1002.0841310240003)
INFO :      configure_evaluate: strategy sampled 3 clients (out of 3)


...

INFO :      aggregate_fit: received 3 results and 0 failures


(ClientAppActor pid=23997) Loss: 0.3850


INFO :      fit progress: (5, 10.617755189538002, {'accuracy': 86.4}, 5280.981167536001)
INFO :      configure_evaluate: strategy sampled 3 clients (out of 3)



GLOBAL MODEL -> Round 5 Accuracy: 86.40%


INFO :      aggregate_evaluate: received 3 results and 0 failures
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished 5 round(s) in 5385.38s
INFO :      	History (loss, distributed):
INFO :      		round 1: 14.85165587067604
INFO :      		round 2: 12.061711803078651
INFO :      		round 3: 11.608558520674706
INFO :      		round 4: 11.354005739092827
INFO :      		round 5: 10.617755189538002
INFO :      	History (loss, centralized):
INFO :      		round 0: 22.157822847366333
INFO :      		round 1: 14.85165587067604
INFO :      		round 2: 12.061711803078651
INFO :      		round 3: 11.608558520674706
INFO :      		round 4: 11.354005739092827
INFO :      		round 5: 10.617755189538002
INFO :      	History (metrics, centralized):
INFO :      	{'accuracy': [(0, 50.0), (1, 80.7), (2, 82.3), (3, 82.4), (4, 84.8), (5, 86.4)]}
INFO :      
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for remov

In [ ]:
import matplotlib.pyplot as plt

rounds = range(
    1,
    len(global_accuracies) + 1
)

plt.figure(figsize=(10,6))

plt.plot(
    rounds,
    global_accuracies,
    marker='o',
    linewidth=2
)

plt.xticks(rounds)

plt.xlabel("Communication Rounds")

plt.ylabel("Global Validation Accuracy (%)")

plt.title("Federated Training Convergence Plot")

plt.grid(True)

plt.show()

In [23]:
import os

print(
    os.path.exists(
        "federated_skin_cancer_model.pth"
    )
)

True


In [24]:
model = SkinCancerCNN().to(device)

model.load_state_dict(
    torch.load(
        "federated_skin_cancer_model.pth",
        map_location=device
    )
)

model.eval()

SkinCancerCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): AdaptiveAvgPool2d(output_size=(1, 1))
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=256, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(

In [31]:
print("Federated global model saved successfully!")

Federated global model saved successfully!


In [32]:
torch.save(

    global_model.state_dict(),

    "federated_skin_cancer_model.pth"
)